# MTR genes Pan

## textfirstmergetext

In [ ]:
"""
genetextannotationprocesstext - textcancer typetext(text)
==============================================

text:
1. textcancer typetextprocess
2. textrowsprocess
3. eachcancer typetextlogfile
4. texteachcancer typetextprocessStep2andStep3
"""

import pandas as pd
import os
import gzip
import json
import re
from collections import defaultdict
from Bio import SeqIO
from multiprocessing import Pool, cpu_count
import time
import sys
from datetime import datetime

# ==================== text ====================
class CancerLogger:
    """foreachcancer typecreatetextrecordtext"""
    def __init__(self, cancer_type, log_dir):
        """
        textrecordtext
        Args:
        cancer_type: cancer typetext
        log_dir: log filessavedirectory
        """
        self.cancer_type = cancer_type
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)
        # createlog files(text)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.log_file = os.path.join(log_dir, f"{cancer_type}_processing_{timestamp}.log")
        # textlog files
        self.file_handle = open(self.log_file, 'w', encoding='utf-8')
        # text
        self.log(f"=" * 80)
        self.log(f"cancer type {cancer_type} processtext")
        self.log(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        self.log(f"=" * 80)
        def log(self, message):
            """
            recordtextinformation
            Args:
            message: textrecordtext
            """
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            log_line = f"[{timestamp}] {message}"
            # textfile
            self.file_handle.write(log_line + '\n')
            self.file_handle.flush() # texttofile
            def close(self):
                """textlog files"""
                self.log(f"\n{'=' * 80}")
                self.log(f"cancer type {self.cancer_type} processend")
                self.log(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                self.log(f"{'=' * 80}")
                self.file_handle.close()
                def __enter__(self):
                    """textwithtext"""
                    return self
                def __exit__(self, exc_type, exc_val, exc_tb):
                    """textwithtext"""
                    if exc_type is not None:
                        self.log(f"\nerrortext: {exc_type.__name__}: {exc_val}")
                        self.close()


# ==================== Global configuration ====================
class GlobalConfig:
    """textallpathandtext"""
    # ===== cancer-type list =====
    CANCER_TYPES = ["CRC", "BRCA", "CESC", "LIHC", "OSCC", "LUNG", "PAAD", "STAD", "THCA"]
    # ===== textPath configuration(textcancer typetext)=====
    BASE_DIR = "/path/to/project/data/genome_annotation"
    BASE_RESULTS_DIR = "/path/to/project/results_V3/cancers_V3.1"
    # ===== textdirectory =====
    LOG_DIR = "/path/to/project/results_V3/cancers_V3.1/processing_logs"
    # ===== inputfile(text)=====
    ASSEMBLY_INFO_FILE = f"{BASE_DIR}/V7_matched_assembly_info.xlsx"
    TAXID_HIERARCHY_FILE = f"{BASE_DIR}/V7_taxid_hierarchy.txt"
    # ===== textannotationfiledirectory(text, text)=====
    GFF_RENAMED_DIR = f"{BASE_DIR}/gff_mic"
    GTF_RENAMED_DIR = f"{BASE_DIR}/gtf_mic"
    # ===== textgenetextFASTAdirectory(text)=====
    GENOME_FASTA_DIR = "/path/to/project/data/reference"
    # ===== processtext =====
    COVERAGE_THRESHOLD = 0
    GENE_MERGE_GAP = 1000 # textadjacent genesgaptext
    @classmethod
    def get_cancer_paths(cls, cancer_type):
        """textcancer typePath configuration"""
        cancer_base = os.path.join(cls.BASE_RESULTS_DIR, cancer_type)
        return {
        'cancer_type': cancer_type,
        'results_dir': os.path.join(cancer_base, "05.MTR/05.2.region"),
        'results_out_dir': os.path.join(cancer_base, "05.MTR/05.3.MTR_genes"),
        'coverage_dir': os.path.join(cancer_base, "03.coverage/Normal"),
        'annotation_output_dir': os.path.join(cancer_base, "05.MTR/05.3.MTR_genes/1.annotation_results"),
        'filtered_output_dir': os.path.join(cancer_base, "05.MTR/05.3.MTR_genes/2.filtered_regions_no_merge"),
        'report_dir': os.path.join(cancer_base, "05.MTR/05.3.MTR_genes/2.filtered_regions_no_merge/report"),
        'log_dir': os.path.join(cancer_base, "05.MTR/05.3.MTR_genes")
    }


# ==================== text2: extractgeneannotation ====================
def parse_gff_line(line):
    """textGFF/GTFrows"""
    if line.startswith('#') or not line.strip():
        return None
    fields = line.strip().split('\t')
    if len(fields) < 9:
        return None
    return {
        'seqid': fields[0],
        'type': fields[2],
        'start': int(fields[3]),
        'end': int(fields[4]),
        'strand': fields[6],
        'attributes': fields[8]
    }

def parse_gff_attributes(attr_string):
    """textGFFformattext"""
    attributes = {}
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            return attributes

def parse_gtf_attributes(attr_string):
    """textGTFformattext"""
    attributes = {}
    pattern = r'(\w+)\s+"([^"]+)"'
    matches = re.findall(pattern, attr_string)
    for key, value in matches:
        attributes[key] = value
        return attributes

def load_annotation_file(file_path, file_type):
    """loadannotationfile"""
    annotations = defaultdict(list)
    open_func = gzip.open if file_path.endswith('.gz') else open
    with open_func(file_path, 'rt') as f:
        for line in f:
            record = parse_gff_line(line)
            if record:
                annotations[record['seqid']].append(record)
                return annotations

def find_overlapping_genes_gff(seqid, start, end, annotations):
    """findtextgene - GFFformat"""
    overlapping = []
    if seqid not in annotations:
        return overlapping
    for record in annotations[seqid]:
        if not (record['end'] < start or record['start'] > end):
            attrs = parse_gff_attributes(record['attributes'])
            gene_info = {
                'type': record['type'],
                'gene_start': record['start'],
                'gene_end': record['end'],
                'strand': record['strand'],
                'gene_id': attrs.get('gene', attrs.get('gene_id', attrs.get('ID', 'N/A'))),
                'gene_name': attrs.get('Name', attrs.get('gene', 'N/A')),
                'product': attrs.get('product', 'N/A'),
                'locus_tag': attrs.get('locus_tag', 'N/A')
            }
            overlapping.append(gene_info)
            return overlapping

def find_overlapping_genes_gtf(seqid, start, end, annotations):
    """findtextgene - GTFformat"""
    overlapping = []
    if seqid not in annotations:
        return overlapping
    for record in annotations[seqid]:
        if not (record['end'] < start or record['start'] > end):
            attrs = parse_gtf_attributes(record['attributes'])
            gene_info = {
                'type': record['type'],
                'gene_start': record['start'],
                'gene_end': record['end'],
                'strand': record['strand'],
                'gene_id': attrs.get('gene_id', 'N/A'),
                'gene_name': attrs.get('gene_name', attrs.get('gene_id', 'N/A')),
                'transcript_id': attrs.get('transcript_id', 'N/A'),
                'gene_biotype': attrs.get('gene_biotype', 'N/A'),
                'product': attrs.get('product', attrs.get('gene_biotype', 'N/A'))
            }
            overlapping.append(gene_info)
            return overlapping

def extract_sample_name(fna_filename):
    """fromFNAfilenameextractsampletext"""
    if '_rna_' in fna_filename:
        return fna_filename.split('_rna_')[0]
    else:
        return fna_filename.split('_')[0] if '_' in fna_filename else fna_filename.replace('.fna', '')

def run_step2_for_cancer(cancer_type, logger):
    """fortextcancer typetextrowstext2"""
    logger.log("\n" + "=" * 70)
    logger.log(f"text2: processcancer type {cancer_type} geneannotation")
    logger.log("=" * 70)
    # textpath
    paths = GlobalConfig.get_cancer_paths(cancer_type)
    annotation_output_dir = paths['annotation_output_dir']
    results_dir = paths['results_dir']
    # checkinput directorytextexists
    if not os.path.exists(results_dir):
        logger.log(f" warning: input directorytextexists {results_dir}")
        return False
    os.makedirs(annotation_output_dir, exist_ok=True)
    def process_annotation_type(annotation_dir, file_type, find_genes_func):
        """processtextannotationfile(GFForGTF)"""
        logger.log(f"\n process {file_type.upper()} annotation")
        if not os.path.exists(annotation_dir):
            logger.log(f" warning: directorytextexists {annotation_dir}")
            return {}
        species_dirs = [d for d in os.listdir(results_dir)             if os.path.isdir(os.path.join(results_dir, d))]
        logger.log(f" found {len(species_dirs)} species")
        sample_results = defaultdict(list)
        species_assembly_map = defaultdict(set)
        for idx, species_name in enumerate(species_dirs, 1):
            logger.log(f" [{idx}/{len(species_dirs)}] {species_name}")
            ann_files = [os.path.join(annotation_dir, f)                 for f in os.listdir(annotation_dir)                 if f.startswith(species_name) and f.endswith('.gz')]
            if not ann_files:
                logger.log(f" not found{file_type.upper()}annotationfile")
                continue
            logger.log(f" found {len(ann_files)} {file_type.upper()}file")
            annotation_data = []
            for ann_file in ann_files:
                annotations = load_annotation_file(ann_file, file_type)
                annotation_data.append((ann_file, annotations))
                species_dir = os.path.join(results_dir, species_name)
                fna_files = [f for f in os.listdir(species_dir) if f.endswith('.fna')]
                logger.log(f" process {len(fna_files)} FNAfile")
                for fna_file in fna_files:
                    sample_name = extract_sample_name(fna_file)
                    fna_path = os.path.join(species_dir, fna_file)
                    with open(fna_path, 'r') as f:
                        for line in f:
                            if line.startswith('>'):
                                header = line.strip().lstrip('>')
                                parts = header.split('_')
                                if len(parts) >= 3:
                                    seqid = '_'.join(parts[:-2])
                                    start = int(float(parts[-2]))
                                    end = int(float(parts[-1]))
                                    found = False
                                    for ann_file, annotations in annotation_data:
                                        genes = find_genes_func(seqid, start, end, annotations)
                                        if genes:
                                            result = {
                                                'sample': sample_name,
                                                'species': species_name,
                                                'fna_file': fna_file,
                                                'region_id': line.strip(),
                                                'seqid': seqid,
                                                'start': start,
                                                'end': end,
                                                'annotation_file': os.path.basename(ann_file),
                                                'overlapping_genes': genes
                                            }
                                            sample_results[sample_name].append(result)
                                            species_assembly_map[species_name].add(os.path.basename(ann_file))
                                            found = True
                                            break
                                        if not found:
                                            result = {
                                                'sample': sample_name,
                                                'species': species_name,
                                                'fna_file': fna_file,
                                                'region_id': line.strip(),
                                                'seqid': seqid,
                                                'start': start,
                                                'end': end,
                                                'annotation_file': 'NOT_FOUND',
                                                'overlapping_genes': []
                                            }
                                            sample_results[sample_name].append(result)
                                            return {
                                            'sample_results': dict(sample_results),
                                            'species_assembly_map': {k: list(v) for k, v in species_assembly_map.items()}
                                        }
                                        # textprocessGFFandGTF
                                        gff_data = process_annotation_type(
                                            GlobalConfig.GFF_RENAMED_DIR,                                             'gff',                                             find_overlapping_genes_gff
)
gtf_data = process_annotation_type(
    GlobalConfig.GTF_RENAMED_DIR,     'gtf',     find_overlapping_genes_gtf
)
# saveresults
logger.log(f"\n save {cancer_type} results")
def save_sample_results(sample_results, file_type):
    """savetextresults"""
    for sample_name, results in sample_results.items():
        json_file = os.path.join(annotation_output_dir,             f"{sample_name}_annotation_results_{file_type}.json")
        with open(json_file, 'w') as f:
            json.dump(results, f, indent=2)
            flat_results = []
            for r in results:
                if r['overlapping_genes']:
                    for gene in r['overlapping_genes']:
                        row = {
                            'sample': r['sample'],
                            'species': r['species'],
                            'fna_file': r['fna_file'],
                            'region_id': r['region_id'],
                            'seqid': r['seqid'],
                            'region_start': r['start'],
                            'region_end': r['end'],
                            'annotation_file': r['annotation_file'],
                            'gene_type': gene['type'],
                            'gene_start': gene['gene_start'],
                            'gene_end': gene['gene_end'],
                            'gene_id': gene.get('gene_id', 'N/A'),
                            'gene_name': gene.get('gene_name', 'N/A'),
                        }
                        if file_type == 'gff':
                            row['product'] = gene.get('product', 'N/A')
                            row['locus_tag'] = gene.get('locus_tag', 'N/A')
                        else:
                            row['transcript_id'] = gene.get('transcript_id', 'N/A')
                            row['gene_biotype'] = gene.get('gene_biotype', 'N/A')
                            row['product'] = gene.get('product', 'N/A')
                            flat_results.append(row)
                            if flat_results:
                                df = pd.DataFrame(flat_results)
                                csv_file = os.path.join(annotation_output_dir,                                     f"{sample_name}_annotation_results_{file_type}.csv")
                                df.to_csv(csv_file, index=False)
                                if gff_data['sample_results']:
                                    save_sample_results(gff_data['sample_results'], 'gff')
                                    mapping_file = os.path.join(annotation_output_dir, "species_assembly_mapping_gff.json")
                                    with open(mapping_file, 'w') as f:
                                        json.dump(gff_data['species_assembly_map'], f, indent=2)
                                        if gtf_data['sample_results']:
                                            save_sample_results(gtf_data['sample_results'], 'gtf')
                                            mapping_file = os.path.join(annotation_output_dir, "species_assembly_mapping_gtf.json")
                                            with open(mapping_file, 'w') as f:
                                                json.dump(gtf_data['species_assembly_map'], f, indent=2)
                                                logger.log(f" OK {cancer_type} text2completed!")
                                                logger.log(f" GFFsample: {len(gff_data['sample_results'])}")
                                                logger.log(f" GTFsample: {len(gtf_data['sample_results'])}")
                                                return True


# ==================== text3: filtertextextractTargettextgene ====================
def load_bedgraph(bedgraph_file):
    """loadbedgraphfile"""
    regions = defaultdict(list)
    if not os.path.exists(bedgraph_file):
        return regions
    with open(bedgraph_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('track'):
                continue
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                chrom = parts[0]
                start = int(parts[1])
                end = int(parts[2])
                value = float(parts[3])
                if value > GlobalConfig.COVERAGE_THRESHOLD:
                    regions[chrom].append((start, end, value))
                    return regions

def check_gene_coverage(gene_start, gene_end, seqid, bedgraph_regions):
    """checkgenetextNormalsampleintextwithcoverage"""
    if seqid not in bedgraph_regions:
        return False
    for bg_start, bg_end, _ in bedgraph_regions[seqid]:
        if not (bg_end <= gene_start or bg_start >= gene_end):
            return True
        return False

def load_genome_sequences(species_name):
    """loadspeciestextgenetextsequence"""
    possible_names = [
        f"{species_name}.fna",
        f"{species_name}.fa",
        f"{species_name}.fasta",
        f"{species_name}_genomic.fna",
    ]
    sequences = {}
    for fname in possible_names:
        fpath = os.path.join(GlobalConfig.GENOME_FASTA_DIR, fname)
        if os.path.exists(fpath):
            if fpath.endswith('.gz'):
                import gzip
                with gzip.open(fpath, 'rt') as f:
                    for record in SeqIO.parse(f, 'fasta'):
                        sequences[record.id] = str(record.seq)
                    else:
                        for record in SeqIO.parse(fpath, 'fasta'):
                            sequences[record.id] = str(record.seq)
                            return sequences
                        return None

def extract_gene_sequence(seqid, gene_start, gene_end, genome_sequences):
    """fromtextgenetextinextractgenesequence"""
    if not genome_sequences:
        return None, 'no_genome'
    if seqid not in genome_sequences:
        return None, f'seqid_not_found_{seqid}'
    sequence = genome_sequences[seqid]
    if gene_start < 1 or gene_end > len(sequence):
        return None, f'out_of_range_{gene_start}_{gene_end}_vs_{len(sequence)}'
    gene_seq = sequence[gene_start-1:gene_end]
    return gene_seq, 'success'

def merge_adjacent_genes(genes, max_gap=1000):
    """textgenetextregion"""
    if not genes:
        return []
    sorted_genes = sorted(genes, key=lambda x: x['gene_start'])
    regions = []
    current_start = sorted_genes[0]['gene_start']
    current_end = sorted_genes[0]['gene_end']
    current_genes = [sorted_genes[0]]
    for gene in sorted_genes[1:]:
        gap = gene['gene_start'] - current_end
        if gap <= max_gap:
            current_end = max(current_end, gene['gene_end'])
            current_genes.append(gene)
        else:
            regions.append((current_start, current_end, current_genes))
            current_start = gene['gene_start']
            current_end = gene['gene_end']
            current_genes = [gene]
            regions.append((current_start, current_end, current_genes))
            return regions

def process_file_type_for_cancer(cancer_type, file_type, logger):
    """fortextcancer typeprocesstextannotationresults"""
    logger.log(f"\n process {cancer_type} {file_type.upper()} results")
    paths = GlobalConfig.get_cancer_paths(cancer_type)
    annotation_output_dir = paths['annotation_output_dir']
    filtered_output_dir = paths['filtered_output_dir']
    report_dir = paths['report_dir']
    coverage_dir = paths['coverage_dir']
    sample_files = [f for f in os.listdir(annotation_output_dir)         if f.endswith(f'_annotation_results_{file_type}.json')]
    if not sample_files:
        logger.log(f" not found{file_type.upper()}resultsfile")
        return
    logger.log(f" found {len(sample_files)} samples")
    for sample_idx, sample_file in enumerate(sample_files, 1):
        sample_name = sample_file.replace(f'_annotation_results_{file_type}.json', '')
        logger.log(f" [{sample_idx}/{len(sample_files)}] sample: {sample_name}")
        json_path = os.path.join(annotation_output_dir, sample_file)
        with open(json_path, 'r') as f:
            annotation_results = json.load(f)
            species_groups = defaultdict(list)
            for result in annotation_results:
                species_groups[result['species']].append(result)
                all_target_genes = []
                summary_stats = []
                extraction_stats = defaultdict(int)
                for species_name, species_results in species_groups.items():
                    bedgraph_file = os.path.join(coverage_dir, species_name, 'coverage_Normal.bedgraph')
                    normal_regions = load_bedgraph(bedgraph_file)
                    genome_sequences = load_genome_sequences(species_name)
                    total_original_genes = 0
                    total_target_genes = 0
                    genes_with_normal_coverage = 0
                    target_specific_genes = []
                    for orig_result in species_results:
                        if not orig_result['overlapping_genes']:
                            continue
                        total_original_genes += len(orig_result['overlapping_genes'])
                        for gene in orig_result['overlapping_genes']:
                            has_normal_coverage = check_gene_coverage(
                                gene['gene_start'],                                 gene['gene_end'],
                                orig_result['seqid'],                                 normal_regions
)
if has_normal_coverage:
    genes_with_normal_coverage += 1
else:
    target_gene = {
        'sample': orig_result['sample'],
        'species': orig_result['species'],
        'seqid': orig_result['seqid'],
        'gene_start': gene['gene_start'],
        'gene_end': gene['gene_end'],
        'gene_length': gene['gene_end'] - gene['gene_start'],
        'strand': gene['strand'],
        'gene_id': gene.get('gene_id', 'N/A'),
        'gene_name': gene.get('gene_name', 'N/A'),
        'gene_type': gene['type'],
        'product': gene.get('product', 'N/A'),
        'annotation_file': orig_result['annotation_file'],
        'original_region': f"{orig_result['seqid']}:{orig_result['start']}-{orig_result['end']}"
    }
    if file_type == 'gtf':
        target_gene['transcript_id'] = gene.get('transcript_id', 'N/A')
        target_gene['gene_biotype'] = gene.get('gene_biotype', 'N/A')
    else:
        target_gene['locus_tag'] = gene.get('locus_tag', 'N/A')
        target_specific_genes.append(target_gene)
        total_target_genes += 1
        if target_specific_genes:
            seqid_groups = defaultdict(list)
            for gene in target_specific_genes:
                seqid_groups[gene['seqid']].append(gene)
                merged_regions = []
                for seqid, genes in seqid_groups.items():
                    regions = merge_adjacent_genes(genes, max_gap=GlobalConfig.GENE_MERGE_GAP)
                    for region_start, region_end, region_genes in regions:
                        merged_regions.append({
                            'sample': sample_name,
                            'species': species_name,
                            'seqid': seqid,
                            'region_start': region_start,
                            'region_end': region_end,
                            'region_length': region_end - region_start,
                            'gene_count': len(region_genes),
                            'genes': region_genes
                        })
                        output_dir = os.path.join(filtered_output_dir, species_name)
                        os.makedirs(output_dir, exist_ok=True)
                        output_fna = os.path.join(output_dir, f"{sample_name}_target_genes_{file_type}.fna")
                        success_count = 0
                        failed_count = 0
                        with open(output_fna, 'w') as fout:
                            for region in merged_regions:
                                sequence, status = extract_gene_sequence(
                                    region['seqid'],
                                    region['region_start'],
                                    region['region_end'],
                                    genome_sequences
)
if sequence and status == 'success':
    header = f">{region['seqid']}_{region['region_start']}_{region['region_end']}"
    fout.write(header + '\n')
    for i in range(0, len(sequence), 60):
        fout.write(sequence[i:i+60] + '\n')
        success_count += 1
    else:
        failed_count += 1
        extraction_stats[status] += 1
        all_target_genes.extend(target_specific_genes)
        summary_stats.append({
            'species': species_name,
            'original_regions': len(species_results),
            'original_genes': total_original_genes,
            'genes_with_normal_coverage': genes_with_normal_coverage,
            'target_specific_genes': total_target_genes,
            'filtered_out_genes': genes_with_normal_coverage
        })
        # saveresults
        os.makedirs(report_dir, exist_ok=True)
        if all_target_genes:
            df_genes = pd.DataFrame(all_target_genes)
            csv_file = os.path.join(report_dir, f"{sample_name}_target_genes_{file_type}.csv")
            df_genes.to_csv(csv_file, index=False)
            if summary_stats:
                df_summary = pd.DataFrame(summary_stats)
                summary_file = os.path.join(report_dir, f"{sample_name}_summary_{file_type}.csv")
                df_summary.to_csv(summary_file, index=False)

def run_step3_for_cancer(cancer_type, logger):
    """fortextcancer typetextrowstext3"""
    logger.log("\n" + "=" * 70)
    logger.log(f"text3: processcancer type {cancer_type} Targettextgeneextract")
    logger.log("=" * 70)
    paths = GlobalConfig.get_cancer_paths(cancer_type)
    # checkinputtextexists
    if not os.path.exists(paths['annotation_output_dir']):
        logger.log(f" warning: annotationresultsdirectorytextexists {paths['annotation_output_dir']}")
        logger.log(f" textrowstext2")
        return False
    os.makedirs(paths['filtered_output_dir'], exist_ok=True)
    os.makedirs(paths['report_dir'], exist_ok=True)
    # textprocessGFFandGTF
    process_file_type_for_cancer(cancer_type, 'gff', logger)
    process_file_type_for_cancer(cancer_type, 'gtf', logger)
    logger.log(f" OK {cancer_type} text3completed!")
    return True


# ==================== text ====================
def process_single_cancer(cancer_type):
    """processtextcancer type(text2and3)- text"""
    # textdirectory
    paths = GlobalConfig.get_cancer_paths(cancer_type)
    log_dir = paths['log_dir']
    # createtextrecordtext
    with CancerLogger(cancer_type, log_dir) as logger:
        logger.log(f"startprocesscancer type: {cancer_type}")
        logger.log(f"log files: {logger.log_file}")
        start_time = time.time()
        try:
            # text2: extractgeneannotation
            step2_success = run_step2_for_cancer(cancer_type, logger)
            if step2_success:
                # text3: filtertextextractTargettextgene
                step3_success = run_step3_for_cancer(cancer_type, logger)
                elapsed = time.time() - start_time
                logger.log(f"\n{cancer_type} Processing completed! text: {elapsed/60:.2f} text")
                # textoutputtextinformation
                print(f"[completed] {cancer_type} - text: {elapsed/60:.2f}text - text: {logger.log_file}")
        except Exception as e:
            logger.log(f"\nerror: {str(e)}")
            import traceback
            logger.log(traceback.format_exc())
            print(f"[error] {cancer_type} processfailed - text: {logger.log_file}")
            raise
        return cancer_type

def main(parallel=True, n_processes=None):
    """
    text
    Args:
    parallel: textrowsprocesstextcancer type
    n_processes: textrowstext(Nonetext)
    """
    print("=" * 80)
    print("textcancer typegeneannotationprocesstext(text)")
    print("=" * 80)
    print(f"textprocesscancer type: {', '.join(GlobalConfig.CANCER_TYPES)}")
    print(f"textrowsprocess: {'text' if parallel else 'text'}")
    # createtextdirectory
    os.makedirs(GlobalConfig.LOG_DIR, exist_ok=True)
    print(f"textdirectory: {GlobalConfig.LOG_DIR}")
    if parallel:
        if n_processes is None:
            n_processes = min(cpu_count(), len(GlobalConfig.CANCER_TYPES))
            print(f"textrowstext: {n_processes}")
            print("=" * 80)
            print("text: eachcancer typedetailedoutputtextsavetextlogfilein")
            print("=" * 80)
            overall_start = time.time()
            if parallel and len(GlobalConfig.CANCER_TYPES) > 1:
                # textrowsprocess
                print(f"\nstarttextrowsprocess {len(GlobalConfig.CANCER_TYPES)} cancer type...")
                with Pool(processes=n_processes) as pool:
                    results = pool.map(process_single_cancer, GlobalConfig.CANCER_TYPES)
                    print(f"\nallcancer typeProcessing completed: {', '.join(results)}")
            else:
                # textrowsprocess
                print("\nstarttextrowsprocess...")
                for cancer_type in GlobalConfig.CANCER_TYPES:
                    process_single_cancer(cancer_type)
                    total_time = time.time() - overall_start
                    print("\n" + "=" * 80)
                    print("textcompleted!")
                    print("=" * 80)
                    print(f"text: {total_time/60:.2f} text ({total_time:.1f} text)")
                    print(f"averageeachcancer type: {total_time/len(GlobalConfig.CANCER_TYPES)/60:.2f} text")
                    print(f"\ntextcancer typedetailedlog filestext:")
                    for cancer_type in GlobalConfig.CANCER_TYPES:
                        paths = GlobalConfig.get_cancer_paths(cancer_type)
                        print(f" {cancer_type}: {paths['log_dir']}")


if __name__ == "__main__":
    # text
    PARALLEL = True # textrowsprocess
    N_PROCESSES = 4 # textrowstext(text)
    # textrows
    main(parallel=PARALLEL, n_processes=N_PROCESSES)

## textmergetext

In [ ]:
"""
genetextannotationprocesstext - textcancer typetext(textgene+textannotation)
==============================================

text:
1. textcancer typetextprocess
2. textrowsprocess
3. eachcancer typetextlogfile
4. Step3text: eachtargettextgenetextoutput
5. GTF/GFFdeduplicateandfiltertext
6. FNA headertextdetailedgeneannotationinformation
"""

import pandas as pd
import os
import gzip
import json
import re
from collections import defaultdict
from Bio import SeqIO
from multiprocessing import Pool, cpu_count
import time
import sys
from datetime import datetime

# ==================== text ====================
class CancerLogger:
    """foreachcancer typecreatetextrecordtext"""
    def __init__(self, cancer_type, log_dir):
        self.cancer_type = cancer_type
        self.log_dir = log_dir
        os.makedirs(log_dir, exist_ok=True)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        self.log_file = os.path.join(log_dir, f"{cancer_type}_processing_{timestamp}.log")
        self.file_handle = open(self.log_file, 'w', encoding='utf-8')
        self.log(f"=" * 80)
        self.log(f"cancer type {cancer_type} processtext")
        self.log(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        self.log(f"=" * 80)
        def log(self, message):
            timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            log_line = f"[{timestamp}] {message}"
            self.file_handle.write(log_line + '\n')
            self.file_handle.flush()
            def close(self):
                self.log(f"\n{'=' * 80}")
                self.log(f"cancer type {self.cancer_type} processend")
                self.log(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                self.log(f"{'=' * 80}")
                self.file_handle.close()
                def __enter__(self):
                    return self
                def __exit__(self, exc_type, exc_val, exc_tb):
                    if exc_type is not None:
                        self.log(f"\nerrortext: {exc_type.__name__}: {exc_val}")
                        self.close()


# ==================== Global configuration ====================
class GlobalConfig:
    """textallpathandtext"""
    # ===== cancer-type list =====
    # CANCER_TYPES = ["CRC", "BRCA", "CESC", "LIHC", "OSCC", "LUNG", "PAAD", "STAD", "BLCA"]
    CANCER_TYPES = ["BLCA"]
    # ===== textPath configuration =====
    BASE_DIR = "/path/to/project/data/genome_annotation"
    BASE_RESULTS_DIR = "/path/to/project/results_V3/cancers_V3.1"
    # ===== textdirectory =====
    LOG_DIR = "/path/to/project/results_V3/cancers_V3.1/processing_logs"
    # ===== inputfile(text)=====
    ASSEMBLY_INFO_FILE = f"{BASE_DIR}/V7_matched_assembly_info.xlsx"
    TAXID_HIERARCHY_FILE = f"{BASE_DIR}/V7_taxid_hierarchy.txt"
    # ===== textannotationfiledirectory(text)=====
    GFF_RENAMED_DIR = f"{BASE_DIR}/gff_mic"
    GTF_RENAMED_DIR = f"{BASE_DIR}/gtf_mic"
    # ===== textgenetextFASTAdirectory =====
    GENOME_FASTA_DIR = "/path/to/project/data/reference"
    # ===== processtext =====
    COVERAGE_THRESHOLD = 0
    MIN_GENE_LENGTH = 50 # minimum gene length(bp)
    # GTF featuretextfilter
    GTF_VALID_FEATURES = {
        'gene', 'transcript', 'exon', 'CDS',         'mRNA', 'lnc_RNA', 'miRNA', 'ncRNA', 'rRNA', 'tRNA'
    }
    @classmethod
    def get_cancer_paths(cls, cancer_type):
        """textcancer typePath configuration"""
        cancer_base = os.path.join(cls.BASE_RESULTS_DIR, cancer_type)
        return {
        'cancer_type': cancer_type,
        'results_dir': os.path.join(cancer_base, "05.MTR/05.2.region"),
        'results_out_dir': os.path.join(cancer_base, "05.MTR/05.3.MTR_genes"),
        'coverage_dir': os.path.join(cancer_base, "03.coverage/Normal"),
        'annotation_output_dir': os.path.join(cancer_base, "05.MTR/05.3.MTR_genes/1.annotation_results"),
        'filtered_output_dir': os.path.join(cancer_base, "05.MTR/05.3.MTR_genes/2.filtered_genes_no_merge"),
        'report_dir': os.path.join(cancer_base, "05.MTR/05.3.MTR_genes/2.filtered_genes_no_merge/report"),
        'log_dir': os.path.join(cancer_base, "05.MTR/05.3.MTR_genes")
    }


# ==================== text2: extractgeneannotation ====================
def parse_gff_line(line):
    """textGFF/GTFrows"""
    if line.startswith('#') or not line.strip():
        return None
    fields = line.strip().split('\t')
    if len(fields) < 9:
        return None
    return {
        'seqid': fields[0],
        'type': fields[2],
        'start': int(fields[3]),
        'end': int(fields[4]),
        'strand': fields[6],
        'attributes': fields[8]
    }

def parse_gff_attributes(attr_string):
    """textGFFformattext"""
    attributes = {}
    for item in attr_string.split(';'):
        item = item.strip()
        if '=' in item:
            key, value = item.split('=', 1)
            attributes[key.strip()] = value.strip()
            return attributes

def parse_gtf_attributes(attr_string):
    """textGTFformattext"""
    attributes = {}
    pattern = r'(\w+)\s+"([^"]+)"'
    matches = re.findall(pattern, attr_string)
    for key, value in matches:
        attributes[key] = value
        return attributes

def load_annotation_file(file_path, file_type):
    """loadannotationfile"""
    annotations = defaultdict(list)
    open_func = gzip.open if file_path.endswith('.gz') else open
    with open_func(file_path, 'rt') as f:
        for line in f:
            record = parse_gff_line(line)
            if record:
                annotations[record['seqid']].append(record)
                return annotations

def find_overlapping_genes_gff(seqid, start, end, annotations):
    """findtextgene - GFFformat"""
    overlapping = []
    if seqid not in annotations:
        return overlapping
    for record in annotations[seqid]:
        if not (record['end'] < start or record['start'] > end):
            attrs = parse_gff_attributes(record['attributes'])
            gene_info = {
                'type': record['type'],
                'gene_start': record['start'],
                'gene_end': record['end'],
                'strand': record['strand'],
                'gene_id': attrs.get('gene', attrs.get('gene_id', attrs.get('ID', 'N/A'))),
                'gene_name': attrs.get('Name', attrs.get('gene', 'N/A')),
                'product': attrs.get('product', 'N/A'),
                'locus_tag': attrs.get('locus_tag', 'N/A')
            }
            overlapping.append(gene_info)
            return overlapping

def find_overlapping_genes_gtf(seqid, start, end, annotations):
    """findtextgene - GTFformat"""
    overlapping = []
    if seqid not in annotations:
        return overlapping
    for record in annotations[seqid]:
        if not (record['end'] < start or record['start'] > end):
            attrs = parse_gtf_attributes(record['attributes'])
            gene_info = {
                'type': record['type'],
                'gene_start': record['start'],
                'gene_end': record['end'],
                'strand': record['strand'],
                'gene_id': attrs.get('gene_id', 'N/A'),
                'gene_name': attrs.get('gene_name', attrs.get('gene_id', 'N/A')),
                'transcript_id': attrs.get('transcript_id', 'N/A'),
                'gene_biotype': attrs.get('gene_biotype', 'N/A'),
                'product': attrs.get('product', attrs.get('gene_biotype', 'N/A'))
            }
            overlapping.append(gene_info)
            return overlapping

def extract_sample_name(fna_filename):
    """fromFNAfilenameextractsampletext"""
    if '_rna_' in fna_filename:
        return fna_filename.split('_rna_')[0]
    else:
        return fna_filename.split('_')[0] if '_' in fna_filename else fna_filename.replace('.fna', '')

def run_step2_for_cancer(cancer_type, logger):
    """fortextcancer typetextrowstext2"""
    logger.log("\n" + "=" * 70)
    logger.log(f"text2: processcancer type {cancer_type} geneannotation")
    logger.log("=" * 70)
    paths = GlobalConfig.get_cancer_paths(cancer_type)
    annotation_output_dir = paths['annotation_output_dir']
    results_dir = paths['results_dir']
    if not os.path.exists(results_dir):
        logger.log(f" warning: input directorytextexists {results_dir}")
        return False
    os.makedirs(annotation_output_dir, exist_ok=True)
    def process_annotation_type(annotation_dir, file_type, find_genes_func):
        """processtextannotationfile(GFForGTF)"""
        logger.log(f"\n process {file_type.upper()} annotation")
        if not os.path.exists(annotation_dir):
            logger.log(f" warning: directorytextexists {annotation_dir}")
            return {}
        species_dirs = [d for d in os.listdir(results_dir)             if os.path.isdir(os.path.join(results_dir, d))]
        logger.log(f" found {len(species_dirs)} species")
        sample_results = defaultdict(list)
        species_assembly_map = defaultdict(set)
        for idx, species_name in enumerate(species_dirs, 1):
            logger.log(f" [{idx}/{len(species_dirs)}] {species_name}")
            ann_files = [os.path.join(annotation_dir, f)                 for f in os.listdir(annotation_dir)                 if f.startswith(species_name) and f.endswith('.gz')]
            if not ann_files:
                logger.log(f" not found{file_type.upper()}annotationfile")
                continue
            logger.log(f" found {len(ann_files)} {file_type.upper()}file")
            annotation_data = []
            for ann_file in ann_files:
                annotations = load_annotation_file(ann_file, file_type)
                annotation_data.append((ann_file, annotations))
                species_dir = os.path.join(results_dir, species_name)
                fna_files = [f for f in os.listdir(species_dir) if f.endswith('.fna')]
                logger.log(f" process {len(fna_files)} FNAfile")
                for fna_file in fna_files:
                    sample_name = extract_sample_name(fna_file)
                    fna_path = os.path.join(species_dir, fna_file)
                    with open(fna_path, 'r') as f:
                        for line in f:
                            if line.startswith('>'):
                                header = line.strip().lstrip('>')
                                parts = header.split('_')
                                if len(parts) >= 3:
                                    seqid = '_'.join(parts[:-2])
                                    start = int(float(parts[-2]))
                                    end = int(float(parts[-1]))
                                    found = False
                                    for ann_file, annotations in annotation_data:
                                        genes = find_genes_func(seqid, start, end, annotations)
                                        if genes:
                                            result = {
                                                'sample': sample_name,
                                                'species': species_name,
                                                'fna_file': fna_file,
                                                'region_id': line.strip(),
                                                'seqid': seqid,
                                                'start': start,
                                                'end': end,
                                                'annotation_file': os.path.basename(ann_file),
                                                'overlapping_genes': genes
                                            }
                                            sample_results[sample_name].append(result)
                                            species_assembly_map[species_name].add(os.path.basename(ann_file))
                                            found = True
                                            break
                                        if not found:
                                            result = {
                                                'sample': sample_name,
                                                'species': species_name,
                                                'fna_file': fna_file,
                                                'region_id': line.strip(),
                                                'seqid': seqid,
                                                'start': start,
                                                'end': end,
                                                'annotation_file': 'NOT_FOUND',
                                                'overlapping_genes': []
                                            }
                                            sample_results[sample_name].append(result)
                                            return {
                                            'sample_results': dict(sample_results),
                                            'species_assembly_map': {k: list(v) for k, v in species_assembly_map.items()}
                                        }
                                        # textprocessGFFandGTF
                                        gff_data = process_annotation_type(
                                            GlobalConfig.GFF_RENAMED_DIR,                                             'gff',                                             find_overlapping_genes_gff
)
gtf_data = process_annotation_type(
    GlobalConfig.GTF_RENAMED_DIR,     'gtf',     find_overlapping_genes_gtf
)
# saveresults
logger.log(f"\n save {cancer_type} results")
def save_sample_results(sample_results, file_type):
    """savetextresults"""
    for sample_name, results in sample_results.items():
        json_file = os.path.join(annotation_output_dir,             f"{sample_name}_annotation_results_{file_type}.json")
        with open(json_file, 'w') as f:
            json.dump(results, f, indent=2)
            flat_results = []
            for r in results:
                if r['overlapping_genes']:
                    for gene in r['overlapping_genes']:
                        row = {
                            'sample': r['sample'],
                            'species': r['species'],
                            'fna_file': r['fna_file'],
                            'region_id': r['region_id'],
                            'seqid': r['seqid'],
                            'region_start': r['start'],
                            'region_end': r['end'],
                            'annotation_file': r['annotation_file'],
                            'gene_type': gene['type'],
                            'gene_start': gene['gene_start'],
                            'gene_end': gene['gene_end'],
                            'gene_id': gene.get('gene_id', 'N/A'),
                            'gene_name': gene.get('gene_name', 'N/A'),
                        }
                        if file_type == 'gff':
                            row['product'] = gene.get('product', 'N/A')
                            row['locus_tag'] = gene.get('locus_tag', 'N/A')
                        else:
                            row['transcript_id'] = gene.get('transcript_id', 'N/A')
                            row['gene_biotype'] = gene.get('gene_biotype', 'N/A')
                            row['product'] = gene.get('product', 'N/A')
                            flat_results.append(row)
                            if flat_results:
                                df = pd.DataFrame(flat_results)
                                csv_file = os.path.join(annotation_output_dir,                                     f"{sample_name}_annotation_results_{file_type}.csv")
                                df.to_csv(csv_file, index=False)
                                if gff_data['sample_results']:
                                    save_sample_results(gff_data['sample_results'], 'gff')
                                    mapping_file = os.path.join(annotation_output_dir, "species_assembly_mapping_gff.json")
                                    with open(mapping_file, 'w') as f:
                                        json.dump(gff_data['species_assembly_map'], f, indent=2)
                                        if gtf_data['sample_results']:
                                            save_sample_results(gtf_data['sample_results'], 'gtf')
                                            mapping_file = os.path.join(annotation_output_dir, "species_assembly_mapping_gtf.json")
                                            with open(mapping_file, 'w') as f:
                                                json.dump(gtf_data['species_assembly_map'], f, indent=2)
                                                logger.log(f" OK {cancer_type} text2completed!")
                                                logger.log(f" GFFsample: {len(gff_data['sample_results'])}")
                                                logger.log(f" GTFsample: {len(gtf_data['sample_results'])}")
                                                return True


# ==================== text3: text ====================
def load_bedgraph(bedgraph_file):
    """loadbedgraphfile"""
    regions = defaultdict(list)
    if not os.path.exists(bedgraph_file):
        return regions
    with open(bedgraph_file, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('track'):
                continue
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                chrom = parts[0]
                start = int(parts[1])
                end = int(parts[2])
                value = float(parts[3])
                if value > GlobalConfig.COVERAGE_THRESHOLD:
                    regions[chrom].append((start, end, value))
                    return regions

def check_gene_coverage(gene_start, gene_end, seqid, bedgraph_regions):
    """checkgenetextNormalsampleintextwithcoverage"""
    if seqid not in bedgraph_regions:
        return False
    for bg_start, bg_end, _ in bedgraph_regions[seqid]:
        if not (bg_end < gene_start or bg_start + 1 > gene_end):
            return True
        return False

def load_genome_sequences(species_name):
    """loadspeciestextgenetextsequence"""
    possible_names = [
        f"{species_name}.fna",
        f"{species_name}.fa",
        f"{species_name}.fasta",
        f"{species_name}_genomic.fna",
    ]
    sequences = {}
    for fname in possible_names:
        fpath = os.path.join(GlobalConfig.GENOME_FASTA_DIR, fname)
        if os.path.exists(fpath):
            if fpath.endswith('.gz'):
                import gzip
                with gzip.open(fpath, 'rt') as f:
                    for record in SeqIO.parse(f, 'fasta'):
                        sequences[record.id] = str(record.seq)
                    else:
                        for record in SeqIO.parse(fpath, 'fasta'):
                            sequences[record.id] = str(record.seq)
                            return sequences
                        return None

def extract_gene_sequence(seqid, gene_start, gene_end, genome_sequences):
    """fromtextgenetextinextractgenesequence"""
    if not genome_sequences:
        return None, 'no_genome'
    if seqid not in genome_sequences:
        return None, f'seqid_not_found_{seqid}'
    sequence = genome_sequences[seqid]
    if gene_start < 1 or gene_end > len(sequence):
        return None, f'out_of_range_{gene_start}_{gene_end}_vs_{len(sequence)}'
    gene_seq = sequence[gene_start-1:gene_end]
    return gene_seq, 'success'

def build_fasta_header(gene, file_type):
    """builddetailedFASTA header"""
    header_parts = [f"{gene['seqid']}:{gene['gene_start']}-{gene['gene_end']}"]
    if file_type == 'gtf':
        gene_id = gene.get('gene_id', 'N/A')
        gene_name = gene.get('gene_name', 'N/A')
        gene_biotype = gene.get('gene_biotype', 'N/A')
        product = gene.get('product', 'N/A')
        if gene_id != 'N/A':
            header_parts.append(f"gene_id={gene_id}")
            if gene_name != 'N/A':
                header_parts.append(f"gene_name={gene_name}")
                if gene_biotype != 'N/A':
                    header_parts.append(f"biotype={gene_biotype}")
                    if product != 'N/A':
                        header_parts.append(f"product={product}")
                    else: # GFFformat
                        locus_tag = gene.get('locus_tag', 'N/A')
                        gene_name = gene.get('gene_name', 'N/A')
                        product = gene.get('product', 'N/A')
                        if locus_tag != 'N/A':
                            header_parts.append(f"locus_tag={locus_tag}")
                            if gene_name != 'N/A':
                                header_parts.append(f"gene={gene_name}")
                                if product != 'N/A':
                                    header_parts.append(f"product={product}")
                                    header_parts.append(f"strand={gene['strand']}")
                                    header_parts.append(f"length={gene['gene_length']}bp")
                                    return ">" + " ".join(header_parts)

def deduplicate_genes_gtf(genes):
    """GTFdeduplicate: textgene_idtext, textfeature"""
    filtered = []
    for gene in genes:
        gene_length = gene['gene_end'] - gene['gene_start']
        if gene_length < GlobalConfig.MIN_GENE_LENGTH:
            continue
        if gene['type'] not in GlobalConfig.GTF_VALID_FEATURES:
            continue
        filtered.append(gene)
        if not filtered:
            return []
        gene_dict = {}
        for gene in filtered:
            gene_id = gene.get('gene_id', 'N/A')
            gene_length = gene['gene_end'] - gene['gene_start']
            if gene_id not in gene_dict:
                gene_dict[gene_id] = gene
            else:
                existing_length = gene_dict[gene_id]['gene_end'] - gene_dict[gene_id]['gene_start']
                if gene_length > existing_length:
                    gene_dict[gene_id] = gene
                    return list(gene_dict.values())

def deduplicate_genes_gff(genes):
    """GFFdeduplicate: textlocus_tagortextdeduplicate"""
    filtered = []
    for gene in genes:
        gene_length = gene['gene_end'] - gene['gene_start']
        if gene_length >= GlobalConfig.MIN_GENE_LENGTH:
            filtered.append(gene)
            if not filtered:
                return []
            gene_dict = {}
            genes_without_locus = []
            for gene in filtered:
                locus_tag = gene.get('locus_tag', None)
                if locus_tag and locus_tag != 'N/A':
                    gene_length = gene['gene_end'] - gene['gene_start']
                    if locus_tag not in gene_dict:
                        gene_dict[locus_tag] = gene
                    else:
                        existing_length = gene_dict[locus_tag]['gene_end'] - gene_dict[locus_tag]['gene_start']
                        if gene_length > existing_length:
                            gene_dict[locus_tag] = gene
                        else:
                            genes_without_locus.append(gene)
                            position_dict = {}
                            for gene in genes_without_locus:
                                key = (gene['gene_start'], gene['gene_end'])
                                gene_length = gene['gene_end'] - gene['gene_start']
                                if key not in position_dict:
                                    position_dict[key] = gene
                                else:
                                    existing_type = position_dict[key]['type']
                                    current_type = gene['type']
                                    priority = {'CDS': 3, 'gene': 2, 'mRNA': 2}
                                    existing_priority = priority.get(existing_type, 1)
                                    current_priority = priority.get(current_type, 1)
                                    if current_priority > existing_priority:
                                        position_dict[key] = gene
                                        result = list(gene_dict.values()) + list(position_dict.values())
                                        return result

def process_file_type_for_cancer(cancer_type, file_type, logger):
    """fortextcancer typeprocesstextannotationresults(text)"""
    logger.log(f"\n process {cancer_type} {file_type.upper()} results(text)")
    paths = GlobalConfig.get_cancer_paths(cancer_type)
    annotation_output_dir = paths['annotation_output_dir']
    filtered_output_dir = paths['filtered_output_dir']
    report_dir = paths['report_dir']
    coverage_dir = paths['coverage_dir']
    sample_files = [f for f in os.listdir(annotation_output_dir)         if f.endswith(f'_annotation_results_{file_type}.json')]
    if not sample_files:
        logger.log(f" not found{file_type.upper()}resultsfile")
        return
    logger.log(f" found {len(sample_files)} samples")
    for sample_idx, sample_file in enumerate(sample_files, 1):
        sample_name = sample_file.replace(f'_annotation_results_{file_type}.json', '')
        logger.log(f" [{sample_idx}/{len(sample_files)}] sample: {sample_name}")
        json_path = os.path.join(annotation_output_dir, sample_file)
        with open(json_path, 'r') as f:
            annotation_results = json.load(f)
            species_groups = defaultdict(list)
            for result in annotation_results:
                species_groups[result['species']].append(result)
                all_target_genes = []
                summary_stats = []
                extraction_stats = defaultdict(int)
                filter_stats = defaultdict(int)
                for species_name, species_results in species_groups.items():
                    bedgraph_file = os.path.join(coverage_dir, species_name, 'coverage_Normal.bedgraph')
                    normal_regions = load_bedgraph(bedgraph_file)
                    genome_sequences = load_genome_sequences(species_name)
                    if genome_sequences:
                        logger.log(f" loadgenetext: {len(genome_sequences)} textsequence")
                        total_original_genes = 0
                        total_after_filter = 0
                        total_target_genes = 0
                        genes_with_normal_coverage = 0
                        target_specific_genes = []
                        for orig_result in species_results:
                            if not orig_result['overlapping_genes']:
                                continue
                            total_original_genes += len(orig_result['overlapping_genes'])
                            # deduplicateandfilter
                            if file_type == 'gtf':
                                valid_genes = deduplicate_genes_gtf(orig_result['overlapping_genes'])
                            else:
                                valid_genes = deduplicate_genes_gff(orig_result['overlapping_genes'])
                                filter_stats['filtered_short_features'] += (len(orig_result['overlapping_genes']) - len(valid_genes))
                                total_after_filter += len(valid_genes)
                                # textcheckgene
                                for gene in valid_genes:
                                    has_normal_coverage = check_gene_coverage(
                                        gene['gene_start'],                                         gene['gene_end'],
                                        orig_result['seqid'],                                         normal_regions
)
if has_normal_coverage:
    genes_with_normal_coverage += 1
else:
    target_gene = {
        'sample': orig_result['sample'],
        'species': orig_result['species'],
        'seqid': orig_result['seqid'],
        'gene_start': gene['gene_start'],
        'gene_end': gene['gene_end'],
        'gene_length': gene['gene_end'] - gene['gene_start'],
        'strand': gene['strand'],
        'gene_id': gene.get('gene_id', 'N/A'),
        'gene_name': gene.get('gene_name', 'N/A'),
        'gene_type': gene['type'],
        'product': gene.get('product', 'N/A'),
        'annotation_file': orig_result['annotation_file'],
        'original_region': f"{orig_result['seqid']}:{orig_result['start']}-{orig_result['end']}"
    }
    if file_type == 'gtf':
        target_gene['transcript_id'] = gene.get('transcript_id', 'N/A')
        target_gene['gene_biotype'] = gene.get('gene_biotype', 'N/A')
    else:
        target_gene['locus_tag'] = gene.get('locus_tag', 'N/A')
        target_specific_genes.append(target_gene)
        total_target_genes += 1
        logger.log(f" species: {species_name}")
        logger.log(f" textgenetext: {total_original_genes}")
        logger.log(f" filterafter: {total_after_filter}")
        logger.log(f" Normalcoverage: {genes_with_normal_coverage}")
        logger.log(f" Targettext: {total_target_genes}")
        # outputFNA
        if target_specific_genes:
            output_dir = os.path.join(filtered_output_dir, species_name)
            os.makedirs(output_dir, exist_ok=True)
            output_fna = os.path.join(output_dir, f"{sample_name}_target_genes_{file_type}.fna")
            success_count = 0
            failed_count = 0
            with open(output_fna, 'w') as fout:
                for gene in target_specific_genes:
                    sequence, status = extract_gene_sequence(
                        gene['seqid'],
                        gene['gene_start'],
                        gene['gene_end'],
                        genome_sequences
)
if sequence and status == 'success':
    header = build_fasta_header(gene, file_type)
    fout.write(header + '\n')
    for i in range(0, len(sequence), 60):
        fout.write(sequence[i:i+60] + '\n')
        success_count += 1
    else:
        failed_count += 1
        extraction_stats[status] += 1
        logger.log(f" FNA: success {success_count}, failed {failed_count}")
        all_target_genes.extend(target_specific_genes)
        summary_stats.append({
            'species': species_name,
            'original_regions': len(species_results),
            'original_genes': total_original_genes,
            'filtered_genes': total_after_filter,
            'genes_with_normal_coverage': genes_with_normal_coverage,
            'target_specific_genes': total_target_genes
        })
        # saveresults
        os.makedirs(report_dir, exist_ok=True)
        if all_target_genes:
            df_genes = pd.DataFrame(all_target_genes)
            df_genes = df_genes.sort_values(['species', 'seqid', 'gene_start'])
            csv_file = os.path.join(report_dir, f"{sample_name}_target_genes_{file_type}.csv")
            df_genes.to_csv(csv_file, index=False)
            if summary_stats:
                df_summary = pd.DataFrame(summary_stats)
                summary_file = os.path.join(report_dir, f"{sample_name}_summary_{file_type}.csv")
                df_summary.to_csv(summary_file, index=False)

def run_step3_for_cancer(cancer_type, logger):
    """fortextcancer typetextrowstext3(text)"""
    logger.log("\n" + "=" * 70)
    logger.log(f"text3: processcancer type {cancer_type} Targettextgeneextract(text)")
    logger.log("=" * 70)
    paths = GlobalConfig.get_cancer_paths(cancer_type)
    if not os.path.exists(paths['annotation_output_dir']):
        logger.log(f" warning: annotationresultsdirectorytextexists {paths['annotation_output_dir']}")
        logger.log(f" textrowstext2")
        return False
    os.makedirs(paths['filtered_output_dir'], exist_ok=True)
    os.makedirs(paths['report_dir'], exist_ok=True)
    # textprocessGFFandGTF
    process_file_type_for_cancer(cancer_type, 'gff', logger)
    process_file_type_for_cancer(cancer_type, 'gtf', logger)
    logger.log(f" OK {cancer_type} text3completed!")
    return True


# ==================== text ====================
def process_single_cancer(cancer_type):
    """processtextcancer type(text2and3)- text"""
    paths = GlobalConfig.get_cancer_paths(cancer_type)
    log_dir = paths['log_dir']
    with CancerLogger(cancer_type, log_dir) as logger:
        logger.log(f"startprocesscancer type: {cancer_type}")
        logger.log(f"log files: {logger.log_file}")
        logger.log(f"text: textgene + textannotation")
        start_time = time.time()
        try:
            # text2: extractgeneannotation
            step2_success = run_step2_for_cancer(cancer_type, logger)
            if step2_success:
                # text3: filtertextextractTargettextgene(text)
                step3_success = run_step3_for_cancer(cancer_type, logger)
                elapsed = time.time() - start_time
                logger.log(f"\n{cancer_type} Processing completed! text: {elapsed/60:.2f} text")
                print(f"[completed] {cancer_type} - text: {elapsed/60:.2f}text - text: {logger.log_file}")
        except Exception as e:
            logger.log(f"\nerror: {str(e)}")
            import traceback
            logger.log(traceback.format_exc())
            print(f"[error] {cancer_type} processfailed - text: {logger.log_file}")
            raise
        return cancer_type

def main(parallel=True, n_processes=None):
    """
    text
    Args:
    parallel: textrowsprocesstextcancer type
    n_processes: textrowstext(Nonetext)
    """
    print("=" * 80)
    print("textcancer typegeneannotationprocesstext(text + textannotation)")
    print("=" * 80)
    print(f"textprocesscancer type: {', '.join(GlobalConfig.CANCER_TYPES)}")
    print(f"textrowsprocess: {'text' if parallel else 'text'}")
    print(f"minimum gene length: {GlobalConfig.MIN_GENE_LENGTH} bp")
    print("\ntext:")
    print(" OK eachtargettextgenetextoutput")
    print(" OK GTF/GFFdeduplicateandfilter")
    print(" OK FNA headertextdetailedannotationinformation")
    print(" OK eachcancer typetextlog files")
    os.makedirs(GlobalConfig.LOG_DIR, exist_ok=True)
    print(f"\ntextdirectory: {GlobalConfig.LOG_DIR}")
    if parallel:
        if n_processes is None:
            n_processes = min(cpu_count(), len(GlobalConfig.CANCER_TYPES))
            print(f"textrowstext: {n_processes}")
            print("=" * 80)
            print("text: eachcancer typedetailedoutputtextsavetextlogfilein")
            print("=" * 80)
            overall_start = time.time()
            if parallel and len(GlobalConfig.CANCER_TYPES) > 1:
                print(f"\nstarttextrowsprocess {len(GlobalConfig.CANCER_TYPES)} cancer type...")
                with Pool(processes=n_processes) as pool:
                    results = pool.map(process_single_cancer, GlobalConfig.CANCER_TYPES)
                    print(f"\nallcancer typeProcessing completed: {', '.join(results)}")
            else:
                print("\nstarttextrowsprocess...")
                for cancer_type in GlobalConfig.CANCER_TYPES:
                    process_single_cancer(cancer_type)
                    total_time = time.time() - overall_start
                    print("\n" + "=" * 80)
                    print("textcompleted!")
                    print("=" * 80)
                    print(f"text: {total_time/60:.2f} text ({total_time:.1f} text)")
                    print(f"averageeachcancer type: {total_time/len(GlobalConfig.CANCER_TYPES)/60:.2f} text")
                    print(f"\ntextcancer typedetailedlog filestext:")
                    for cancer_type in GlobalConfig.CANCER_TYPES:
                        paths = GlobalConfig.get_cancer_paths(cancer_type)
                        print(f" {cancer_type}: {paths['log_dir']}")


if __name__ == "__main__":
    # text
    PARALLEL = True # textrowsprocess
    N_PROCESSES = 64 # textrowstext(text)
    # textrows
    main(parallel=PARALLEL, n_processes=N_PROCESSES)